In [3]:
!pip install langchain_google_genai pinecone

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached websockets-16.0-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   --------------------------- ------------ 524.3/750.9 kB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 750.9/750.9 kB 3.7 MB/s  0:00:00
Using cached websockets-16.0-cp313-cp313-win_amd64.whl (178 kB)
   ---------------------------------------- 0.0/742.7 kB ? eta -:--:--
   ---------------------------------------- 742.7/742.7 kB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------  3.4/3.5 MB 25.3 MB/s eta 0:00:01
   -----


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pinecone import Pinecone
import os
import json
from dotenv import load_dotenv
load_dotenv()
# Setup
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", output_dimensionality=768)
 
# Sample question
question = "what is diabetes"
embedded_query = embed_model.embed_query(question)
 
# Query Pinecone
result = index.query(vector=embedded_query, top_k=3, include_metadata=True)
 
print("\n" + "="*80)
print("PINECONE METADATA DEBUG")
print("="*80)
 
for i, match in enumerate(getattr(result, "matches", [])):
    print(f"\n--- MATCH {i} ---")
    print(f"Match ID: {match.get('id', 'N/A')}")
    print(f"Score: {match.get('score', 'N/A')}")
    
    metadata = match.get("metadata", {})
    print(f"\nMetadata Keys: {list(metadata.keys())}")
    
    print(f"\nFull Metadata:")
    for key, value in metadata.items():
        value_preview = str(value)[:100]
        print(f"  {key}: {value_preview}...")
    
    # Check common field names
    print(f"\nField Length Check:")
    for field in ["text", "content", "page_content", "body", "data", "doc"]:
        value = metadata.get(field, "NOT FOUND")
        if value != "NOT FOUND":
            print(f"  {field}: {len(str(value))} chars")
        else:
            print(f"  {field}: NOT FOUND")
 
print("\n" + "="*80)


PINECONE METADATA DEBUG

--- MATCH 0 ---
Match ID: DIABETES-0
Score: 0.751368821

Metadata Keys: ['author', 'creationdate', 'creator', 'moddate', 'page', 'page_label', 'producer', 'source', 'total_pages']

Full Metadata:
  author: Jyothi...
  creationdate: D:20121023153348...
  creator: Microsoft® Office Word 2007...
  moddate: D:20121023153348...
  page: 0...
  page_label: 1...
  producer: Microsoft® Office Word 2007...
  source: uploaded_docs\DIABETES.pdf...
  total_pages: 5...

Field Length Check:
  text: NOT FOUND
  content: NOT FOUND
  page_content: NOT FOUND
  body: NOT FOUND
  data: NOT FOUND
  doc: NOT FOUND

--- MATCH 1 ---
Match ID: DIABETES-1
Score: 0.731416881

Metadata Keys: ['author', 'creationdate', 'creator', 'moddate', 'page', 'page_label', 'producer', 'source', 'total_pages']

Full Metadata:
  author: Jyothi...
  creationdate: D:20121023153348...
  creator: Microsoft® Office Word 2007...
  moddate: D:20121023153348...
  page: 0...
  page_label: 1...
  producer: Micro